In [ ]:
import os
import json
import pandas as pd

def dataset_transfer(file_path, new_path):
    messages = []

    input_text = ""
    entity_sentence = ""

    tokens = []
    tags = []

    current_token = []
    current_type = None

    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                if tokens:
                    input_text = "".join(tokens)

                    for token, tag in zip(tokens, tags):
                        if tag.startswith("B-"):
                            if current_token:
                                entity_obj = {"entity_text": "".join(current_token), "entity_label": current_type}
                                entity_sentence += json.dumps(entity_obj, ensure_ascii=False) + "\n"
                            current_token = [token]
                            current_type = tag[2:]
                        elif tag.startswith("I-"):
                            current_token.append(token)
                        else:
                            if current_token:
                                entity_obj = {"entity_text": "".join(current_token), "entity_label": current_type}
                                entity_sentence += json.dumps(entity_obj, ensure_ascii=False) + "\n"
                            current_token = []
                            current_type = None
                        
                    if current_token and current_type:
                        entity_obj = {"entity_text": "".join(current_token), "entity_label": current_type}
                        entity_sentence += json.dumps(entity_obj, ensure_ascii=False) + "\n"
                        current_token = []
                        current_type = None

                    if entity_sentence == "":
                        entity_sentence = "没有找到任何实体"
                    
                    message = {
                        "instruction": """你是一个中医药领域的文本实体识别的专家，你需要从给定的句子中提取 临床表现; 中医诊断; 西医诊断; 西医治疗; 中药; 方剂; 中医治则 等等中医药领域实体. 以 json 格式输出, 如 {"entity_text": "肉桂", "entity_label": "中药"} 注意: 1. 输出的每一行都必须是正确的 json 字符串. 2. 找不到任何实体时, 输出"没有找到任何实体". """,
                        "input": f"文本:{input_text}",
                        "output": entity_sentence,
                    }

                    messages.append(message)

                    tokens = []
                    tags = []
                    entity_sentence = ""

            else:
                parts = line.split()
                if len(parts) >= 2:
                    tokens.append(parts[0])
                    tags.append(parts[-1])
        
        if tokens:
            input_text = "".join(tokens)

            for token, tag in zip(tokens, tags):
                if tag.startswith("B-"):
                    if current_token:
                        entity_obj = {"entity_text": "".join(current_token), "entity_label": current_type}
                        entity_sentence += json.dumps(entity_obj, ensure_ascii=False) + "\n"
                    current_token = [token]
                    current_type = tag[2:]
                elif tag.startswith("I-"):
                    current_token.append(token)
                else:
                    if current_token:
                        entity_obj = {"entity_text": "".join(current_token), "entity_label": current_type}
                        entity_sentence += json.dumps(entity_obj, ensure_ascii=False) + "\n"
                    current_token = []
                    current_type = None
            if current_token and current_type:
                entity_obj = {"entity_text": "".join(current_token), "entity_label": current_type}
                entity_sentence += json.dumps(entity_obj, ensure_ascii=False) + "\n"
                current_token = []
                current_type = None

            if entity_sentence == "":
                entity_sentence = "没有找到任何实体"
            
            message = {
                "instruction": """你是一个中医药领域的文本实体识别的专家，你需要从给定的句子中提取 临床表现; 中医诊断; 西医诊断; 西医治疗; 中药; 方剂; 中医治则 等等中医药领域实体. 以 json 格式输出, 如 {"entity_text": "肉桂", "entity_label": "中药"} 注意: 1. 输出的每一行都必须是正确的 json 字符串. 2. 找不到任何实体时, 输出"没有找到任何实体". """,
                "input": f"文本:{input_text}",
                "output": entity_sentence,
            }

            messages.append(message)

    with open(new_path, "w", encoding="utf-8") as file:
        for message in messages:
            file.write(json.dumps(message, ensure_ascii=False) + "\n")

            

train_dataset_path = "./medical.train"
test_dataset_path = "./medical.test"
dev_dataset_path = "./medical.dev"

train_jsonl_new_path = "./medical_train.jsonl"
test_jsonl_new_path = "./medical_test.jsonl"
dev_jsonl_new_path = "./medical_dev.jsonl"

if not os.path.exists(train_jsonl_new_path):
    dataset_transfer(train_dataset_path, train_jsonl_new_path)
if not os.path.exists(test_jsonl_new_path):
    dataset_transfer(test_dataset_path, test_jsonl_new_path)
if not os.path.exists(dev_jsonl_new_path):
    dataset_transfer(dev_dataset_path, dev_jsonl_new_path)

train_df = pd.read_json(train_jsonl_new_path,lines=True)
test_df = pd.read_json(test_jsonl_new_path, lines=True)
dev_df = pd.read_json(dev_jsonl_new_path, lines=True)


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, DataCollatorForSeq2Seq, Trainer
import torch
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-7B")

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id
    
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-7B",
                                             torch_dtype=torch.bfloat16)
messages = [
    {"role": "user", "content": "Who are you?"},
]

inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(model.device)

outputs = model.generate(**inputs, max_new_tokens=40)
print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:]))

In [ ]:
def process_func(example):
    MAX_LENGTH = 768
    
    messages = [
        {"role": "system", "content": example['instruction']},
        {"role": "user", "content": example['input']}
    ]
    

    prompt_ids = tokenizer.apply_chat_template(
        messages, 
        tokenize=True, 
        add_generation_prompt=True,  
        return_tensors=None
    )
    
    response_ids = tokenizer(
        example['output'] + tokenizer.eos_token, 
        add_special_tokens=False
    )['input_ids']
    
    input_ids = prompt_ids + response_ids
    attention_mask = [1] * len(input_ids)
    labels = [-100] * len(prompt_ids) + response_ids
    
    if len(input_ids) > MAX_LENGTH:
        input_ids = input_ids[:MAX_LENGTH]
        attention_mask = attention_mask[:MAX_LENGTH]
        labels = labels[:MAX_LENGTH]
        
    return {
        "input_ids": input_ids, 
        "attention_mask": attention_mask, 
        "labels": labels
    }

In [ ]:
from datasets import Dataset

train_ds = Dataset.from_pandas(train_df)
train_dataset = train_ds.map(process_func, remove_columns=train_ds.column_names)

dev_ds = Dataset.from_pandas(dev_df)
dev_dataset = dev_ds.map(process_func, remove_columns=dev_ds.column_names)

In [ ]:
from peft import LoraConfig, TaskType, get_peft_model

config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    inference_mode=False, 
    r=8,  
    lora_alpha=32,  
    lora_dropout=0.1, 
)

model = get_peft_model(model, config)
model.enable_input_require_grads()

In [ ]:
import re
import json
import numpy as np
from collections import defaultdict


args = TrainingArguments(
    output_dir="./output/Qwen2-NER",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    num_train_epochs=2,
    
    eval_strategy="epoch",       
    save_strategy="epoch",  
    save_total_limit=2,
    
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    learning_rate=1e-4,
    gradient_checkpointing=True,
    bf16=True,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    data_collator=DataCollatorForSeq2Seq(tokenizer, padding=True, label_pad_token_id=-100),
)

trainer.train()


In [ ]:
import torch

def parse_json_entities(text):
    if not text or "没有找到任何实体" in text:
        return []
    pattern = r'\{[^{}]*?"entity_text"[^{}]*?"entity_label"[^{}]*?\}'
    matches = re.findall(pattern, text, re.DOTALL)
    entities = []
    for m in matches:
        try:
            obj = json.loads(m)
            entities.append({
                "text": obj["entity_text"].strip(),
                "label": obj["entity_label"].strip(),
            })
        except json.JSONDecodeError:
            continue
    return entities

def compute_f1(all_gold, all_pred):
    tp = fp = fn = 0
    for g, p in zip(all_gold, all_pred):
        g_set = {(e["text"], e["label"]) for e in g}
        p_set = {(e["text"], e["label"]) for e in p}
        tp += len(g_set & p_set)
        fp += len(p_set - g_set)
        fn += len(g_set - p_set)
    
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    
    return {"precision": precision, "recall": recall, "f1": f1}


@torch.no_grad()
def evaluate_generation(model, df, batch_size=4, max_new_tokens=256):
    model.eval()
    tokenizer.padding_side = "left" 

    all_gold, all_pred = [], []
    device = next(model.parameters()).device

    for start in range(0, len(df), batch_size):
        batch_df = df.iloc[start:start + batch_size]

        prompts = []
        for _, row in batch_df.iterrows():
            messages = [
                {"role": "system", "content": row["instruction"]},
                {"role": "user", "content": row["input"]},
            ]
            prompt = tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True
            )
            prompts.append(prompt)

        gold_texts = [row["output"] for _, row in batch_df.iterrows()]

        inputs = tokenizer(
            prompts, return_tensors="pt",
            padding=True, truncation=True, max_length=768
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}

        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,  
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

        gen_ids = outputs[:, inputs["input_ids"].shape[1]:]
        pred_texts = tokenizer.batch_decode(gen_ids, skip_special_tokens=True)

        all_gold.extend([parse_json_entities(t) for t in gold_texts])
        all_pred.extend([parse_json_entities(t) for t in pred_texts])
    
    tokenizer.padding_side = "right"

    return compute_f1(all_gold, all_pred)


# 训练完成后调用
dev_f1 = evaluate_generation(trainer.model, dev_df)
print(f"Dev F1: {dev_f1}")

test_f1 = evaluate_generation(trainer.model, test_df)
print(f"Test F1: {test_f1}")